# 05 — Central and temporal EEG: separate calibration and evaluation

Central means verified FC/C/CP labels. Temporal FT/T/TP labels are reported
separately with independent fits. Klados uses paper-list row order as an
explicit assumption; paired core-central rows are C3/C4/Cz. Native OSF uses
actual named channels and separate whole-trial calibration/scoring blocks.

Compare identity, joint HEOG/VEOG ridge (penalty0.001) and GEVD–MWF (rank1),
with fixed correction strengths0.25/1. No parameter search, VMD, neural training,
frontal communication, LEMON or confirmation-data access. No posterior correction.


## 1. Exact published source and fresh environment

In [ ]:
RUN_SPEC = {'run_id': 'central-temporal-001', 'campaign_id': 'central-temporal-20261017', 'kernel': 'satyapaladugu/central-temporal-eog-comparison', 'git_sha': 'RESOLVED_TO_PUBLISHED_SHA_AT_SUBMISSION', 'git_url': 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git', 'created_utc': '2026-10-17T06:45:59.236083+00:00', 'output': '/kaggle/working/results/central-temporal-20261017/central-temporal-001'}
import os,sys,pathlib,tempfile,subprocess,json,platform
assert pathlib.Path('/kaggle/input').exists(), 'Execute numerical reporting on Kaggle'
os.environ.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
print('BOOTSTRAP_START',RUN_SPEC['run_id'],RUN_SPEC['git_sha'],flush=True)
CHECKOUT=pathlib.Path(tempfile.mkdtemp(prefix='central-source-'))
subprocess.run(['git','clone','--quiet',RUN_SPEC['git_url'],str(CHECKOUT)],check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']],check=True)
actual=subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
assert actual==RUN_SPEC['git_sha']
subprocess.run([sys.executable,'-m','pip','install','-q','pytest==8.3.5','pymatreader==1.0.0',
    'tabulate==0.9.0','torchinfo==1.8.0','PySDKit==0.5.0','condat-tv==0.0.5'],check=True,timeout=300)
sys.path.insert(0,str(CHECKOUT/'src'))
import torch,importlib.metadata
torch.set_num_threads(2)
from eog_clean.data import save_json
OUTPUT=pathlib.Path(RUN_SPEC['output']);OUTPUT.mkdir(parents=True,exist_ok=True)
save_json(OUTPUT/'run_spec.json',RUN_SPEC)
save_json(OUTPUT/'environment.json',{'git_sha':actual,'python':sys.version,'platform':platform.platform(),
    'cuda':torch.cuda.is_available(),'versions':{name:importlib.metadata.version(name) for name in
    ['numpy','scipy','pandas','torch','matplotlib']}})


## 2. Contracts and numerical experiment

All numerical work occurs here on Kaggle. Check disjoint anatomical groups,
fixed calibration, single subtraction and the existing signal/metric contracts.
Original source archives/checksums and the exact archived Klados record split
must verify. Each completed record/group saves recoverable predictions.


In [ ]:
environment=os.environ.copy();environment['PYTHONPATH']=str(CHECKOUT/'src')
tests=subprocess.run([sys.executable,'-m','pytest','-q',str(CHECKOUT/'tests'),
    '--junitxml='+str(OUTPUT/'contract_tests.xml')],text=True,capture_output=True,env=environment)
(OUTPUT/'contract_tests.txt').write_text(tests.stdout+'\n'+tests.stderr)
print(tests.stdout,flush=True)
if tests.returncode:raise RuntimeError('Contracts failed; stop before processing')
from eog_clean.central import run,report
run('/kaggle/input',CHECKOUT,OUTPUT)
FIGURES=report(OUTPUT)


## 3. Results, preservation and limitations

Paired SNR/RMSE/CC are Klados only. Native before/after HEOG/VEOG correlations
use unchanged references; resting preservation is a proxy without clean truth.
Absolute coefficients are averaged before participant aggregation. Failed fits
remain identity outputs and are disclosed. A preservation pass alone does not
establish suppression. Klados is historically exposed and row binding unverified.
Calibration/filter boundaries differ from the archived frontal comparison;
regional SNRs cannot be added or treated as matched superiority estimates.


In [ ]:
import pandas as pd
from IPython.display import display,Image,Markdown
for name in ['klados_summary.csv','osf_summary.csv','osf_rest_summary.csv','preservation_gates.csv']:
    print(name);display(pd.read_csv(OUTPUT/name).round(6))
print((OUTPUT/'completion.json').read_text())
print((OUTPUT/'calibration_failures.json').read_text())
for path in FIGURES:display(Image(filename=path))
save_json(OUTPUT/'execution_state.json',{'state':'complete','git_sha':actual,
    'groups':['central','temporal'],'frontal_context_used':False,'production_qualified':False})
